<a href="https://colab.research.google.com/github/Samarjamal326/Flyrank/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-10 Review

**Lane:** CTR / Engagement Opportunity Scoring

This notebook freezes one transparent, non-fitted baseline before model work. The rule uses only information available at the **2026-02-28 decision cutoff** and writes a ranked review queue.

The baseline is intentionally simple and readable: it combines **CTR relative to position** with **search volume**, then assigns one reason code and one action label. The later ML model must beat this frozen rule on the same evaluation definition.

## 1. My rule and its reason codes

### Signal 1 — CTR versus position

**Why:** FlyRank's CTR-fix logic is position-aware: a low CTR is more concerning when a page already has meaningful search visibility. We first inspect February CTR by position tier.

**Verdict:** the bucket table will determine this from the measured data; the notebook prints the verdict rather than hard-coding it.

### Signal 2 — Search volume / impressions

**Why:** CTR is noisy when exposure is tiny. A page with enough impressions gives a more useful review candidate. This is the volume signal behind the quick-win logic.

**Verdict:** the notebook checks zero-click / low-CTR behavior across impression buckets before the rule is encoded.

### Frozen rule

A page enters the review queue only if it has **at least 100 measured February impressions** and a valid position. Its score is:

`opportunity_score = positive_ctr_gap × log1p(feb_impressions)`

where `positive_ctr_gap` is the amount by which the page's February CTR is below the median CTR of its own February position tier.

This is not a fitted model: there are no learned weights.

- **Reason code:** `below_position_ctr`
- **Action:** `Review CTR opportunity`
- If the page is not below its position-tier benchmark, it is not prioritized.

The rule deliberately uses only February information. March is reserved for the outcome check and never enters the score.

### Setup

The warehouse is gated. Put the approved Hugging Face READ token in Colab Secrets as `HF_TOKEN`; never paste the token into this notebook or commit it.

We use February as the feature/decision window and March only as an observed outcome window, matching the ML-04 contract.

In [8]:
!pip -q install duckdb pandas numpy

import os
import duckdb
import pandas as pd
import numpy as np

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. Add it in Colab Secrets and enable Notebook access."
    )

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])
con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN getvariable('hf_token'))"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Warehouse connected.")


Warehouse connected.


### Build the February decision frame

Only measured GSC rows are used. The position is impression-weighted. The IDs remain keys for joining and review; they are not model inputs.

In [9]:
feb = con.sql(f"""
WITH agg AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS feb_impressions,
        SUM(gsc_clicks) FILTER (WHERE gsc_data_available IS TRUE) AS feb_clicks,
        SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE) AS feb_sum_position,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS feb_measured_days
    FROM {FEB}
    GROUP BY 1, 2
)
SELECT
    client_hash_id,
    content_hash_id,
    feb_impressions,
    feb_clicks,
    feb_clicks / NULLIF(feb_impressions, 0) AS feb_ctr,
    feb_sum_position / NULLIF(feb_impressions, 0) AS feb_avg_position,
    feb_measured_days,
    CASE
        WHEN feb_sum_position / NULLIF(feb_impressions, 0) <= 3 THEN 'top_3'
        WHEN feb_sum_position / NULLIF(feb_impressions, 0) <= 10 THEN 'page_1'
        WHEN feb_sum_position / NULLIF(feb_impressions, 0) <= 20 THEN 'striking'
        WHEN feb_sum_position / NULLIF(feb_impressions, 0) <= 50 THEN 'page_3_5'
        ELSE 'deep'
    END AS position_tier
FROM agg
WHERE feb_measured_days > 0
  AND feb_impressions >= 100
  AND feb_sum_position > 0
""").df()

print(f"February decision rows: {len(feb):,}")
display(feb.head())


February decision rows: 80,321


,client_hash_id,content_hash_id,feb_impressions,feb_clicks,feb_ctr,feb_avg_position,feb_measured_days,position_tier
0,client_3ffa76342f366962,content_dd66eecf9626cab8,235.0,0.0,0.000000,6.391489,28,page_1
1,client_3ffa76342f366962,content_456ab2db28595187,100.0,2.0,0.020000,4.200000,25,page_1
2,client_3ffa76342f366962,content_5573434837db89c5,198.0,6.0,0.030303,7.318182,24,page_1
3,client_3ffa76342f366962,content_7b17975c58745266,102.0,5.0,0.049020,4.450980,26,page_1
4,client_3ffa76342f366962,content_b89167cd03d6ffc1,178.0,6.0,0.033708,3.393258,26,page_1


### Signal check 1 — CTR by position tier

`n` is printed for every bucket. The verdict is based on whether the measured February CTR generally changes in the expected direction as position worsens. This is descriptive evidence, not a causal claim.

In [10]:
position_buckets = (
    feb.groupby("position_tier", observed=True)
       .agg(
           n=("content_hash_id", "size"),
           median_ctr=("feb_ctr", "median"),
           mean_ctr=("feb_ctr", "mean"),
           median_position=("feb_avg_position", "median"),
           median_impressions=("feb_impressions", "median"),
       )
       .reindex(["top_3", "page_1", "striking", "page_3_5", "deep"])
)

display(position_buckets.round(3))

ordered = position_buckets.dropna(subset=["median_ctr"])
if len(ordered) >= 3:
    diffs = np.diff(ordered["median_ctr"].to_numpy())
    position_verdict = "CONFIRMED" if np.sum(diffs <= 0) >= max(1, len(diffs)-1) else "MIXED"
else:
    position_verdict = "MIXED"

print(f"Verdict — CTR versus position: {position_verdict}")


,n,median_ctr,mean_ctr,median_position,median_impressions
position_tier,,,,,
top_3,12057,0.002,0.003,2.091,1080.0
page_1,40903,0.002,0.003,5.846,835.0
striking,15911,0.001,0.002,13.569,475.0
page_3_5,10239,0.000,0.001,27.570,459.0
deep,1211,0.000,0.001,58.994,188.0


Verdict — CTR versus position: CONFIRMED


### Signal check 2 — volume / impression bucket

The rule needs enough exposure for CTR differences to be useful. We therefore inspect CTR and zero-click rates across measured February impression buckets.

The bucket table is also a guard against pretending that a single click on a tiny denominator is a stable opportunity signal.

In [11]:
feb["impression_bucket"] = pd.cut(
    feb["feb_impressions"],
    bins=[99, 249, 499, 999, 4999, np.inf],
    labels=["100-249", "250-499", "500-999", "1k-4.9k", "5k+"],
    right=True,
)

volume_buckets = (
    feb.groupby("impression_bucket", observed=False)
       .agg(
           n=("content_hash_id", "size"),
           median_impressions=("feb_impressions", "median"),
           median_ctr=("feb_ctr", "median"),
           zero_click_rate=("feb_clicks", lambda s: (s == 0).mean()),
       )
)

display(volume_buckets.round(4))

# The rule only needs a stability floor, so a clear decline in zero-click rate
# as exposure increases is not required for confirmation.
zero_rates = volume_buckets["zero_click_rate"].dropna().to_numpy()
if len(zero_rates) >= 3:
    volume_verdict = "CONFIRMED" if zero_rates[0] >= zero_rates[-1] else "MIXED"
else:
    volume_verdict = "MIXED"

print(f"Verdict — volume as a stability signal: {volume_verdict}")


,n,median_impressions,median_ctr,zero_click_rate
impression_bucket,,,,
100-249,18763,159.0,0.0000,0.7520
250-499,14425,353.0,0.0000,0.5764
500-999,13826,697.0,0.0015,0.3582
1k-4.9k,24937,2026.0,0.0021,0.1078
5k+,8370,8498.5,0.0024,0.0092


Verdict — volume as a stability signal: CONFIRMED


## 2. Build the ranked queue

The benchmark is calculated **within the February decision window only**, separately for each position tier.

The score rewards two things:
1. a larger observed CTR shortfall versus comparable-position pages;
2. enough impressions for that shortfall to matter.

There are no fitted coefficients and no March inputs.

In [12]:
import os

os.makedirs("work/outputs", exist_ok=True)
print("Output directory ready:", os.path.abspath("work/outputs"))

Output directory ready: /content/work/outputs


In [13]:
# Position-tier benchmark: February only.
tier_benchmark = (
    feb.groupby("position_tier", observed=True)["feb_ctr"]
       .median()
       .rename("tier_median_ctr")
)

queue = feb.merge(
    tier_benchmark,
    on="position_tier",
    how="left",
)

queue["ctr_gap_pp"] = (
    queue["tier_median_ctr"] - queue["feb_ctr"]
).clip(lower=0)

queue["opportunity_score"] = (
    queue["ctr_gap_pp"] * np.log1p(queue["feb_impressions"])
)

queue["reason_code"] = np.where(
    queue["ctr_gap_pp"] > 0,
    "below_position_ctr",
    "not_below_position_benchmark",
)

queue["action"] = np.where(
    queue["ctr_gap_pp"] > 0,
    "Review CTR opportunity",
    "Monitor",
)

queue = queue.sort_values(
    ["opportunity_score", "feb_impressions"],
    ascending=[False, False],
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

output_cols = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "opportunity_score",
    "action",
    "reason_code",
    "position_tier",
    "feb_avg_position",
    "feb_impressions",
    "feb_clicks",
    "feb_ctr",
    "tier_median_ctr",
    "ctr_gap_pp",
]

queue[output_cols].to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False,
)

print(f"Ranked queue rows: {len(queue):,}")
print("Wrote: work/outputs/baseline_action_score.csv")
display(queue[output_cols].head(10).round(4))


Ranked queue rows: 80,321
Wrote: work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,opportunity_score,action,reason_code,position_tier,feb_avg_position,feb_impressions,feb_clicks,feb_ctr,tier_median_ctr,ctr_gap_pp
0,1,client_73cda7b4e4f265ea,content_fec55986a1868d62,0.0230,Review CTR opportunity,below_position_ctr,top_3,0.0703,193954.0,0.0,0.0000,0.0019,0.0019
1,2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,0.0229,Review CTR opportunity,below_position_ctr,top_3,0.2595,203401.0,2.0,0.0000,0.0019,0.0019
2,3,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,0.0229,Review CTR opportunity,below_position_ctr,top_3,0.0099,195648.0,1.0,0.0000,0.0019,0.0019
3,4,client_73cda7b4e4f265ea,content_c9f840183215651b,0.0221,Review CTR opportunity,below_position_ctr,top_3,2.3083,125035.0,0.0,0.0000,0.0019,0.0019
4,5,client_23a62021009f63c4,content_2ac8c7995de53cd1,0.0211,Review CTR opportunity,below_position_ctr,top_3,0.0396,92128.0,4.0,0.0000,0.0019,0.0018
5,6,client_3197e6291363b4db,content_22588e765b93dfac,0.0199,Review CTR opportunity,below_position_ctr,page_1,7.1627,54938.0,2.0,0.0000,0.0019,0.0018
6,7,client_23a62021009f63c4,content_44f34c0a90047651,0.0199,Review CTR opportunity,below_position_ctr,top_3,0.5185,90223.0,13.0,0.0001,0.0019,0.0017
7,8,client_861cdcccf8049915,content_c406f6bcaac8a477,0.0189,Review CTR opportunity,below_position_ctr,page_1,4.2258,30545.0,1.0,0.0000,0.0019,0.0018
8,9,client_73cda7b4e4f265ea,content_1cb7263083e97ba1,0.0185,Review CTR opportunity,below_position_ctr,top_3,0.6264,18472.0,0.0,0.0000,0.0019,0.0019
9,10,client_73cda7b4e4f265ea,content_d16bbebfbb3c8fda,0.0182,Review CTR opportunity,below_position_ctr,top_3,0.1219,15238.0,0.0,0.0000,0.0019,0.0019


## 3. Top-10 review

For every top-ten candidate, the review records:
- the action;
- the reason;
- a confidence note based on volume;
- what would make the pick wrong.

This is a human review, not an assertion that the rule caused anything.

In [14]:
top10 = queue.head(10).copy()

def confidence_note(row):
    if row["feb_impressions"] >= 1000:
        return "higher exposure; CTR estimate is more stable than the minimum-volume cases"
    if row["feb_impressions"] >= 500:
        return "moderate exposure; still review denominator"
    return "minimum/low exposure; inspect carefully"

def wrong_if(row):
    return (
        "wrong if the position-tier benchmark is not comparable for this content, "
        "the measured February exposure is incomplete, or the CTR gap is driven by "
        "small-denominator noise / an unobserved context."
    )

review = top10[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action",
        "reason_code",
        "position_tier",
        "feb_impressions",
        "feb_ctr",
        "tier_median_ctr",
        "ctr_gap_pp",
    ]
].copy()

review["confidence_note"] = top10.apply(confidence_note, axis=1).values
review["what_would_make_it_wrong"] = top10.apply(wrong_if, axis=1).values

display(review.round(4))


,rank,client_hash_id,content_hash_id,action,reason_code,position_tier,feb_impressions,feb_ctr,tier_median_ctr,ctr_gap_pp,confidence_note,what_would_make_it_wrong
0,1,client_73cda7b4e4f265ea,content_fec55986a1868d62,Review CTR opportunity,below_position_ctr,top_3,193954.0,0.0000,0.0019,0.0019,higher exposure; CTR estimate is more stable t...,wrong if the position-tier benchmark is not co...
1,2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,Review CTR opportunity,below_position_ctr,top_3,203401.0,0.0000,0.0019,0.0019,higher exposure; CTR estimate is more stable t...,wrong if the position-tier benchmark is not co...
2,3,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,Review CTR opportunity,below_position_ctr,top_3,195648.0,0.0000,0.0019,0.0019,higher exposure; CTR estimate is more stable t...,wrong if the position-tier benchmark is not co...
3,4,client_73cda7b4e4f265ea,content_c9f840183215651b,Review CTR opportunity,below_position_ctr,top_3,125035.0,0.0000,0.0019,0.0019,higher exposure; CTR estimate is more stable t...,wrong if the position-tier benchmark is not co...
4,5,client_23a62021009f63c4,content_2ac8c7995de53cd1,Review CTR opportunity,below_position_ctr,top_3,92128.0,0.0000,0.0019,0.0018,higher exposure; CTR estimate is more stable t...,wrong if the position-tier benchmark is not co...
5,6,client_3197e6291363b4db,content_22588e765b93dfac,Review CTR opportunity,below_position_ctr,page_1,54938.0,0.0000,0.0019,0.0018,higher exposure; CTR estimate is more stable t...,wrong if the position-tier benchmark is not co...
6,7,client_23a62021009f63c4,content_44f34c0a90047651,Review CTR opportunity,below_position_ctr,top_3,90223.0,0.0001,0.0019,0.0017,higher exposure; CTR estimate is more stable t...,wrong if the position-tier benchmark is not co...
7,8,client_861cdcccf8049915,content_c406f6bcaac8a477,Review CTR opportunity,below_position_ctr,page_1,30545.0,0.0000,0.0019,0.0018,higher exposure; CTR estimate is more stable t...,wrong if the position-tier benchmark is not co...
8,9,client_73cda7b4e4f265ea,content_1cb7263083e97ba1,Review CTR opportunity,below_position_ctr,top_3,18472.0,0.0000,0.0019,0.0019,higher exposure; CTR estimate is more stable t...,wrong if the position-tier benchmark is not co...
9,10,client_73cda7b4e4f265ea,content_d16bbebfbb3c8fda,Review CTR opportunity,below_position_ctr,top_3,15238.0,0.0000,0.0019,0.0019,higher exposure; CTR estimate is more stable t...,wrong if the position-tier benchmark is not co...


## 4. Weak picks + leakage check

The baseline should have at least one candidate worth questioning. A weak pick is not a failure; finding it is the point of the review.

We also verify that the ranking columns contain no March outcome and no label-derived fields.

In [15]:
weak_pick = queue.sort_values(
    ["opportunity_score", "feb_impressions"],
    ascending=[False, False]
).tail(1)

print("Example of a weak/least-prioritized candidate:")
display(
    weak_pick[
        [
            "rank",
            "content_hash_id",
            "action",
            "reason_code",
            "feb_impressions",
            "feb_ctr",
            "tier_median_ctr",
            "ctr_gap_pp",
        ]
    ].round(4)
)

forbidden = {
    "future_ctr",
    "mar_impressions",
    "mar_clicks",
    "trend_direction",
    "trend_pct",
}

used_for_score = {
    "feb_impressions",
    "feb_ctr",
    "tier_median_ctr",
    "ctr_gap_pp",
    "opportunity_score",
}

assert not used_for_score & forbidden
assert "future_ctr" not in queue.columns
assert "mar_impressions" not in queue.columns
assert "mar_clicks" not in queue.columns

print("Leakage check: PASS — score uses February-only observed inputs.")


Example of a weak/least-prioritized candidate:


,rank,content_hash_id,action,reason_code,feb_impressions,feb_ctr,tier_median_ctr,ctr_gap_pp
80320,80321,content_7f9455b6f25fef37,Monitor,not_below_position_benchmark,100.0,0.0,0.0,0.0


Leakage check: PASS — score uses February-only observed inputs.


## 5. Self-check

- [x] Two signal checks have visible bucket tables and `n`.
- [x] At least one signal is directly linked to FlyRank's CTR-fix / position reasoning.
- [x] Volume is used as a stability floor, not as a claim of causality.
- [x] One transparent score is frozen before model work.
- [x] Every scored item has one reason code and one action label.
- [x] Ranked queue is written to `work/outputs/baseline_action_score.csv`.
- [x] Top 10 has action, reason, confidence note, and what would make it wrong.
- [x] No future-window or label-derived fields enter the score.
- [x] Claims are framed as observed, measured, directional, and decision-support.
- [ ] Run top-to-bottom and commit `work/notebooks/w04_baseline_score.ipynb`.
